# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Jericho-Ram/FlyRank-Internship-ML/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Lane:** content refresh / decline risk -- rank existing pages by how likely they are to lose
search clicks next month, so a content or SEO editor knows what to open first.

**The question:** across an editor's queue of already-published pages, which ones are declining
in search performance right now, and which of those are worth a refresh versus a data-quality
check versus nothing at all?

**Decision it supports:** what a content team opens first this sprint, out of a queue too large
to review by hand (26,604 scorable pages in this dataset). **Who acts on it:** a content/SEO
editor doing the actual review -- this work produces a ranked list and reasons, not an automated
edit.

**Cost of a wrong call, both directions.** A false positive sends an editor to refresh a page
that was never declining -- wasted editorial hours on a page that didn't need it. A false
negative lets a genuinely declining page sit unreviewed while it keeps losing clicks. Neither
error is free, which is why Section 4 reports precision at several queue depths (not just one)
and Section 6's playbook separates "act now" from "investigate" rather than collapsing both into
one score.

**What changed since Week 1.** The original framing (`w01_research_question.ipynb`) asked this
question in the abstract. Week 2 (`w02_ml_task_framing.ipynb`) picked ranking/scoring as the ML
task and quoted early, provisional numbers (baseline 0.240 vs. model 0.680 on an early metric
definition) before the rule and the validation design were rebuilt. Those early numbers are
superseded here by the honest, re-measured versions in Sections 3-4 -- the question stayed the
same, the rigor around answering it did not.

In [1]:
import os
import numpy as np
import pandas as pd

if os.getcwd().replace("\\", "/").endswith("work/notebooks"):
    os.chdir("../..")

CSV_PATH = "data/raw/content_refresh_anonymized.csv"
assert os.path.exists(CSV_PATH), "starter CSV not found -- are you at the repo root?"
df = pd.read_csv(CSV_PATH)

print(f"rows: {len(df):,} | columns: {df.shape[1]} | clients: {df['client_id'].nunique()}")
print(f"declining label rate (raw, unfiltered): {df['trend_direction'].str.lower().eq('down').mean():.3f}")
print(f"rows with avg_position > 0 (actually ranked): {(df['avg_position'] > 0).sum():,}")

rows: 30,000 | columns: 44 | clients: 32
declining label rate (raw, unfiltered): 0.542
rows with avg_position > 0 (actually ranked): 28,795


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

This project used **two** data sources across the ten weeks, for two different jobs, and this
paper's final numbers come from only one of them -- keeping that straight matters (see
`outputs/known_failure_modes.md` entry 5 for what goes wrong when it isn't).

**1. The bundled anonymized starter CSV** -- `data/raw/content_refresh_anonymized.csv`, 30,000
rows, 44 columns, one row per content item. This is the source for every number in Sections
3-7 below: the model, the baseline rule, the leakage audit, and the ranked queue. The label,
`trend_direction`, is `down` when 30-day impressions fell against the prior 30 days
(`trend_pct`, recomputed and verified to match in `w05_model.ipynb`). No client names, URLs, or
raw queries -- `client_id`/`content_id` are opaque hashes.

**2. The FlyRank warehouse** -- `hf://datasets/FlyRank/internship-warehouse`, build `v20260703`,
queried directly in `w03_data_contract.ipynb` and `w03_feature_leakage_check.ipynb`. Features
from calendar month `2026-03` only, label (`is_declining = clicks_apr < clicks_mar`) from
`2026-04` only -- a genuine past-to-future split, unlike the CSV's same-window label. 176,737
rows raw. **This source was used to develop and stress-test the leakage-hunting method, not to
produce this paper's headline numbers** -- its label is defined on `clicks`, the CSV's on
`impressions`, and the two are not comparable (known-failure-mode entry 5). The final month,
`2026-06`, stayed sealed throughout as the natural outcome window of any past-to-future label.

**Excluded from the CSV pipeline, and why:** `trend_direction`, `trend_pct`,
`impressions_last_30d`, `impressions_prev_30d` -- the label and its raw material (Section 3
proves what happens if any of these leak in). `content_id`/`client_id` -- identifiers, not
signal. Rows with `avg_position <= 0` (unranked) and rows with `impressions_prev_30d == 0`
(decline is arithmetically impossible, checked live below) are scored separately, not folded
into this population.

In [2]:
# --- Rows this paper's scored population excludes, and why (checked live) ---
unranked = (df["avg_position"] <= 0).sum()
cannot_decline = (df["avg_position"] > 0) & (df["impressions_prev_30d"] == 0)
print(f"unranked rows (avg_position <= 0), excluded         : {unranked:,}")
print(f"ranked rows where decline is impossible, excluded   : {cannot_decline.sum():,}")

ranked = df["avg_position"] > 0
can_decline = df["impressions_prev_30d"] > 0
pop = df[ranked & can_decline].reset_index(drop=True).copy()
print(f"\nscored population (this paper's Sections 3-7)       : {len(pop):,} rows, "
      f"{pop['client_id'].nunique()} of {df['client_id'].nunique()} clients")
print(f"declining rate in scored population                 : "
      f"{pop['trend_direction'].str.lower().eq('down').mean():.3f}")

unranked rows (avg_position <= 0), excluded         : 1,205
ranked rows where decline is impossible, excluded   : 2,191

scored population (this paper's Sections 3-7)       : 26,604 rows, 31 of 32 clients


declining rate in scored population                 : 0.611


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

**Label.** `is_declining = trend_direction == 'down'`, itself defined from `trend_pct`
(30-day-vs-prior-30-day impression change, verified to correlate 1.000 with a from-scratch
recomputation in `w05_model.ipynb`). Not a forecast label -- both halves of the comparison sit
inside the same 90-day extract, a limitation carried into Section 5.

**Baseline (the rule a human would use without a model).** A page is worth reviewing if it earns
a smaller share of the clicks its position tier normally delivers. Each tier's expected CTR is
measured, not assumed (weighted clicks/impressions per `position_tier`); a page's score is its
missed-click share of that expectation, scored only when expected clicks clear a floor of 5 (below
that, a single real zero-click page and a real underperformer are statistically indistinguishable
-- `w04_baseline_score.ipynb` Section 1 measures this directly).

**Features.** Two sets, both built once and reused everywhere below: **permissive** (38 columns
-- everything that is not the label, an identifier, or a rule-derivation column) and **strict**
(34 columns -- permissive minus `clicks_last_30d`/`clicks_prev_30d`/`sessions_last_30d`/
`sessions_prev_30d`, which share a time window with the label's own impression comparison even
though they are not the label itself).

**Validation design -- grouped by client, and here is why that matters, measured, not asserted.**
The population spans 31 clients (32 in the raw CSV; one drops out entirely under the exclusions
above), and one client alone is 26.2% of the rows -- a random split would let a model learn
*which client this is* instead of the actual signal. The cell below fits the same RandomForest
under both a client-grouped split and a naive random split and reports the gap live.

**Leakage checks, run fresh below:** (A) assert none of the four label-derived columns reached
the feature list: (B) add the label's own raw material (`trend_pct`) back as a feature and watch
the score break; (C) is the grouped-vs-random gap real, not assumed; the fourth and fifth checks
from the original five-test hunt are cited rather than re-run here to keep this notebook's
runtime reasonable -- both were already measured on this exact feature set in
`w06_validation_audit.ipynb`: **(D)** including the 2,191 rows where decline is arithmetically
impossible inflates AUC by **+0.041** (0.645 -> 0.686); **(E)** the four window-overlap columns
alone carry **+0.034** AUC (0.645 permissive vs. 0.611 strict) -- real predictive value, but
partly riding on same-window correlation rather than a leading indicator.

In [3]:
import numpy as np
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

SEED = 42
np.random.seed(SEED)

y = pop["trend_direction"].str.lower().eq("down").to_numpy()
groups = pop["client_id"].to_numpy()

# --- Week-4 rule, re-scored on this population --------------------------------
tier_stats = df[ranked].groupby("position_tier").agg(ti=("impressions_90d", "sum"), tc=("clicks_90d", "sum"))
TIER_CTR = tier_stats["tc"] / tier_stats["ti"]
EXPECTED_CLICKS_FLOOR = 5.0
IN_SCOPE_TIERS = ["page_1", "striking", "page_3_5", "top_3"]
pop["tier_expected_ctr"] = pop["position_tier"].map(TIER_CTR)
pop["expected_clicks"] = pop["impressions_90d"] * pop["tier_expected_ctr"]
pop["missed_clicks"] = pop["expected_clicks"] - pop["clicks_90d"]
_in_scope = pop["position_tier"].isin(IN_SCOPE_TIERS)
_scorable = _in_scope & (pop["expected_clicks"] >= EXPECTED_CLICKS_FLOOR)
_fires = _scorable & (pop["missed_clicks"] > 0)
pop["baseline_score"] = np.where(_fires, pop["missed_clicks"] / pop["expected_clicks"], 0.0)
pop["rule_in_scope"] = _in_scope

LEAK_COLS = ["trend_direction", "trend_pct", "impressions_last_30d", "impressions_prev_30d"]
WINDOW_COLS = ["clicks_last_30d", "clicks_prev_30d", "sessions_last_30d", "sessions_prev_30d"]
DROP_ALWAYS = (["content_id", "client_id"] + LEAK_COLS
               + ["tier_expected_ctr", "expected_clicks", "missed_clicks", "baseline_score", "rule_in_scope"])
FEATURES = [c for c in pop.columns if c not in DROP_ALWAYS]
STRICT_FEATURES = [c for c in FEATURES if c not in WINDOW_COLS]

print("--- Test A: no label-derived column in FEATURES ---")
leaky_present = [c for c in FEATURES if c in LEAK_COLS]
assert not leaky_present, f"leaky columns present: {leaky_present}"
print(f"PASS | permissive: {len(FEATURES)} cols | strict: {len(STRICT_FEATURES)} cols")


def build_pipe(model, cols):
    num = [c for c in cols if pd.api.types.is_numeric_dtype(pop[c])]
    cat = [c for c in cols if c not in num]
    pre = ColumnTransformer([
        ("num", Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler())]), num),
        ("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), cat),
    ])
    return Pipeline([("pre", pre), ("model", model)])


GKF = GroupKFold(n_splits=5)
SKF = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
folds = list(GKF.split(pop, y, groups))
print(f"fold 0 is a single client ({pop.iloc[folds[0][1]]['client_id'].nunique()} client, "
      f"{len(folds[0][1]):,} rows) -- read it as one client's result, not an average")


def cv_rf(cols, splitter=None):
    X = pop[cols]
    it = folds if splitter is None else list(splitter.split(X, y))
    oof = np.zeros(len(pop))
    aucs = []
    for tr, te in it:
        rf = RandomForestClassifier(n_estimators=300, min_samples_leaf=5, n_jobs=-1, random_state=SEED)
        pipe = build_pipe(rf, cols)
        pipe.fit(X.iloc[tr], y[tr])
        p = pipe.predict_proba(X.iloc[te])[:, 1]
        oof[te] = p
        aucs.append(roc_auc_score(y[te], p))
    return oof, aucs


print("\nfitting: permissive (grouped) ...")
oof_permissive, auc_permissive_folds = cv_rf(FEATURES)
print("fitting: strict (grouped) ...")
oof_strict, auc_strict_folds = cv_rf(STRICT_FEATURES)
print("fitting: permissive (random split, Test C) ...")
_, auc_random_folds = cv_rf(FEATURES, splitter=SKF)
print("fitting: permissive + trend_pct (Test B) ...")
_, auc_leaky_folds = cv_rf(FEATURES + ["trend_pct"])

auc_grouped = np.mean(auc_permissive_folds)
auc_strict = np.mean(auc_strict_folds)
auc_random = np.mean(auc_random_folds)
auc_leaky = np.mean(auc_leaky_folds)

print("\n--- Test B: label's raw material added back ---")
print(f"honest (permissive) : {auc_grouped:.3f}")
print(f"+ trend_pct          : {auc_leaky:.3f}  (jump {auc_leaky - auc_grouped:+.3f})")

print("\n--- Test C: grouped vs. random split ---")
print(f"grouped (what ships) : {auc_grouped:.3f}  {[round(a,3) for a in auc_permissive_folds]}")
print(f"random               : {auc_random:.3f}  {[round(a,3) for a in auc_random_folds]}")
print(f"gap                  : {auc_random - auc_grouped:+.3f}")

print(f"\nstrict (window cols dropped) : {auc_strict:.3f}  (vs. permissive {auc_grouped:.3f}, "
      f"value of window cols {auc_grouped - auc_strict:+.3f})")

--- Test A: no label-derived column in FEATURES ---
PASS | permissive: 38 cols | strict: 34 cols
fold 0 is a single client (1 client, 6,981 rows) -- read it as one client's result, not an average

fitting: permissive (grouped) ...


fitting: strict (grouped) ...


fitting: permissive (random split, Test C) ...


fitting: permissive + trend_pct (Test B) ...



--- Test B: label's raw material added back ---
honest (permissive) : 0.645
+ trend_pct          : 1.000  (jump +0.355)

--- Test C: grouped vs. random split ---
grouped (what ships) : 0.645  [0.684, 0.626, 0.624, 0.629, 0.664]
random               : 0.767  [0.76, 0.767, 0.773, 0.768, 0.767]
gap                  : +0.122

strict (window cols dropped) : 0.611  (vs. permissive 0.645, value of window cols +0.034)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Same 5 client-grouped folds for every row below -- the rule needs no fit, LogReg is fit fresh
here, and RandomForest reuses the out-of-fold predictions from Section 3 rather than re-fitting
(identical folds, identical seed, so the numbers are exactly the ones just measured).

RandomForest/permissive is what shipped: **0.645 mean ROC AUC**, against **0.532** for the
Week-4 hand rule -- a real, modest gap over a rule that already knows the rows it should flag on.
Precision@50 tells a different part of the story: RandomForest/**strict** actually reads slightly
*higher* at the top of the queue (0.868) than permissive (0.844), even though its overall AUC is
lower (0.611 vs 0.645) -- a reminder from Week 5 that AUC and precision-at-K can disagree, and
why both are reported rather than picking the one that looks best.

In [4]:
from sklearn.linear_model import LogisticRegression


def precision_at_k(scores, truth, k):
    order = np.argsort(-scores, kind="stable")
    return truth[order[:k]].mean()


KS = [10, 50, 100, 500]
records = []
for fi, (tr, te) in enumerate(folds):
    y_te = y[te]
    records.append({
        "arm": "baseline (Week-4 rule)", "features": "--", "fold": fi,
        "roc_auc": roc_auc_score(y_te, pop.iloc[te]["baseline_score"].to_numpy()),
        **{f"p@{k}": precision_at_k(pop.iloc[te]["baseline_score"].to_numpy(), y_te, k) for k in KS},
    })
    records.append({
        "arm": "RandomForest", "features": "permissive", "fold": fi,
        "roc_auc": roc_auc_score(y_te, oof_permissive[te]),
        **{f"p@{k}": precision_at_k(oof_permissive[te], y_te, k) for k in KS},
    })
    records.append({
        "arm": "RandomForest", "features": "strict", "fold": fi,
        "roc_auc": roc_auc_score(y_te, oof_strict[te]),
        **{f"p@{k}": precision_at_k(oof_strict[te], y_te, k) for k in KS},
    })
    for set_name, cols in [("permissive", FEATURES), ("strict", STRICT_FEATURES)]:
        pipe = build_pipe(LogisticRegression(max_iter=2000, random_state=SEED), cols)
        pipe.fit(pop[cols].iloc[tr], y[tr])
        p = pipe.predict_proba(pop[cols].iloc[te])[:, 1]
        records.append({
            "arm": "LogReg", "features": set_name, "fold": fi,
            "roc_auc": roc_auc_score(y_te, p),
            **{f"p@{k}": precision_at_k(p, y_te, k) for k in KS},
        })

per_fold = pd.DataFrame(records)
metric_cols = ["roc_auc"] + [f"p@{k}" for k in KS]
summary = per_fold.groupby(["arm", "features"], sort=False)[metric_cols].mean().round(3)
print("--- model vs baseline: mean across 5 client-grouped folds ---")
print(summary.to_string())

# --- Chart 1: ROC AUC, baseline vs LogReg vs RandomForest (permissive) --------
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

os.makedirs("work/outputs/charts", exist_ok=True)
chart_rows = summary.reset_index()
chart_rows = chart_rows[(chart_rows["features"] == "permissive") | (chart_rows["arm"] == "baseline (Week-4 rule)")]
order = ["baseline (Week-4 rule)", "LogReg", "RandomForest"]
chart_rows = chart_rows.set_index("arm").loc[order].reset_index()
colors = ["#2a78d6", "#eb6834", "#1baf7a"]

fig, ax = plt.subplots(figsize=(6, 4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
bars = ax.bar(chart_rows["arm"], chart_rows["roc_auc"], color=colors, width=0.55)
ax.axhline(0.5, color="#8a8a86", linestyle="--", linewidth=1)
ax.text(2.5, 0.51, "chance (0.5)", ha="right", color="#52514e", fontsize=9)
for bar, val in zip(bars, chart_rows["roc_auc"]):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 0.015, f"{val:.3f}",
            ha="center", va="bottom", color="#0b0b0b", fontsize=10, fontweight="bold")
ax.set_ylim(0, 0.85)
ax.set_ylabel("Mean ROC AUC (5 client-grouped folds)", color="#0b0b0b")
ax.set_title("Model vs. baseline: does the model beat a hand-written rule?", color="#0b0b0b", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#c3c2b7")
ax.tick_params(colors="#0b0b0b")
fig.text(0.5, -0.03, "RandomForest reaches 0.645 mean AUC vs. 0.532 for the Week-4 rule -- real but modest "
                       "separation from a coin flip.", ha="center", fontsize=9, color="#52514e", wrap=True)
fig.tight_layout()
fig.savefig("work/outputs/charts/results_comparison.svg", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close(fig)
print("\nwrote work/outputs/charts/results_comparison.svg")

--- model vs baseline: mean across 5 client-grouped folds ---
                                   roc_auc  p@10   p@50  p@100  p@500
arm                    features                                      
baseline (Week-4 rule) --            0.532  0.76  0.808  0.782  0.746
RandomForest           permissive    0.645  0.92  0.844  0.842  0.787
                       strict        0.611  0.84  0.868  0.812  0.771
LogReg                 permissive    0.625  0.74  0.796  0.760  0.740
                       strict        0.586  0.72  0.724  0.706  0.704



wrote work/outputs/charts/results_comparison.svg


## 5. Limitations

*What this work cannot claim.*

**Client memorization is real, measured, and mostly what separates the two feature sets.**
Section 3's grouped-vs-random gap (+0.122) means a real share of the model's apparent skill is
recognizing which client a row belongs to, not predicting decline. For a client **not** in this
31-client population, only something closer to the non-memorized share of that skill should be
assumed to carry over.

**Not clean of same-window correlation.** Test E (cited from `w06_validation_audit.ipynb`): the
four window-overlap columns carry +0.034 AUC on their own -- real value, but partly riding on
columns measured over the same window as the label rather than a leading indicator a decision-
maker would have in hand before that window closes. At the row level (checked live below), the
average permissive-vs-strict probability gap is small (+0.004), but it is not evenly spread --
15.3% of rows (4,083) shift by 0.05 or more between the two feature sets, meaning the AUC cost
is concentrated in a minority of rows rather than shared thinly across all of them.

**One fold is one client, not an average.** Fold 0 of the grouped split is a single client
(6,981 rows) -- its per-fold number is that client's result, read as the noisiest fold rather
than weighted equally against the other four.

**Missing data is structured, not random (cited from `w04_signal_audit.ipynb`).** `word_count`
is missing on 25.7% of the raw CSV, concentrated in one client (81.8% missing, 7,008 rows) and
in low-ranking pages (49.3% missing at `deep` vs. 11.5% at `top_3`) -- any test on the non-missing
rows is measuring a sample biased toward already-better-performing content.

**`days_since_last_update` cannot support a staleness claim.** 57 distinct values cover 30,000
rows, five of which cover 87.8% of them -- batch timestamps, not per-page freshness
(`w04_signal_audit.ipynb` Signal 3; the exact distinct count is checked live below).
Nothing in this paper's model or rule uses it as a freshness signal.

**Cross-sectional, not causal, not a forecast.** Every number here describes one 90-day extract.
"The model ranks pages by measured decline probability" is defensible; "refreshing these pages
will fix decline" is not -- no intervention was run, only an association was measured (see
Section 6's honest-claims framing).

**Two datasets, two labels, not comparable.** The warehouse leakage-hunt (Section 2) used a
clicks-based, past-to-future label on a different, larger population; nothing here averages or
compares its numbers directly against the CSV-based results.

In [5]:
# --- Grounding numbers for the limitations above, checked live -------------
print(f"clients in scored population : {pop['client_id'].nunique()} of {df['client_id'].nunique()} raw")
byc = pop.groupby("client_id").size().sort_values()
print(f"largest client                : {byc.iloc[-1]:,} rows ({byc.iloc[-1] / len(pop) * 100:.1f}% of population)")
print(f"smallest client                : {byc.iloc[0]:,} rows")
print(f"clients with < 30 rows here    : {(byc < 30).sum()}")

window_reliance = oof_permissive - oof_strict
print(f"\nmean(permissive - strict) OOF probability gap : {window_reliance.mean():+.4f}")
print(f"rows where permissive - strict >= 0.05         : {(window_reliance >= 0.05).sum():,} "
      f"({(window_reliance >= 0.05).mean() * 100:.1f}%)")

wc_missing = df["word_count"].isna().mean()
print(f"\nword_count missing, raw CSV     : {wc_missing * 100:.1f}%")
dsu_nunique = df["days_since_last_update"].nunique()
print(f"days_since_last_update, distinct values across {len(df):,} rows : {dsu_nunique}")

clients in scored population : 31 of 32 raw
largest client                : 6,981 rows (26.2% of population)
smallest client                : 2 rows
clients with < 30 rows here    : 4

mean(permissive - strict) OOF probability gap : +0.0044
rows where permissive - strict >= 0.05         : 4,083 (15.3%)

word_count missing, raw CSV     : 25.7%
days_since_last_update, distinct values across 30,000 rows : 57


## 6. Ranked recommendations

*The action playbook output -- the paper's recommendations section.*

Reusing the exact out-of-fold model scores and rule scores from Sections 3-4 (no re-fitting), the
same tiering built in `w07_action_playbook.ipynb`: rule and model in agreement rank highest, rows
where the rule is silent but the model is confident are flagged for investigation rather than
automatic action, and rows the rule flags but the model doubts stay in the queue because the two
signals answer different questions (is this page under-clicking its position vs. is it trending
down).

**Read this as decision-support, not confirmation.** The rule-silent/model-flags tier reads a
70.2% true-decline rate in this snapshot -- directional evidence the model adds signal the rule
misses, not proof any specific refresh will work. Full no-go list and human-review checklist are
in `w07_action_playbook.ipynb` Section 3; this section carries the ranked output only.

In [6]:
VISIBLE_TIERS = ["page_1", "top_3", "striking"]
q = pop[["content_id", "client_id", "position_tier", "rule_in_scope", "baseline_score"]].copy()
q["truth_declining"] = y
q["p_decline"] = oof_permissive
q["p_decline_strict"] = oof_strict
q["rule_flags"] = q["baseline_score"] > 0.0
q["model_flags"] = q["p_decline"] >= 0.5


def action_tier(row):
    if row["rule_flags"] and row["model_flags"]:
        return "1_refresh_now"
    if (not row["rule_flags"]) and row["p_decline"] >= 0.6:
        return "2_investigate_rule_silent"
    if row["rule_flags"] and not row["model_flags"]:
        return "3_ctr_gap_only"
    return "4_monitor"


q["action_tier"] = q.apply(action_tier, axis=1)
tier_counts = q["action_tier"].value_counts().sort_index()
print("--- action tier counts ---")
print(tier_counts.to_string())

t2 = q[q["action_tier"] == "2_investigate_rule_silent"]
print(f"\ntier 2 (rule silent, model flags) true-decline rate : {t2['truth_declining'].mean():.3f}")

q_sorted = q.sort_values(["action_tier", "baseline_score"], ascending=[True, False]).reset_index(drop=True)
q_sorted.insert(0, "rank", np.arange(1, len(q_sorted) + 1))

os.makedirs("work/outputs", exist_ok=True)
export_cols = ["rank", "content_id", "client_id", "action_tier", "baseline_score",
               "p_decline", "p_decline_strict", "position_tier", "truth_declining"]
q_sorted[export_cols].to_csv("work/outputs/capstone_action_queue.csv", index=False)
print(f"wrote work/outputs/capstone_action_queue.csv -- {len(q_sorted):,} rows ranked")

# --- Chart 2: action tier mix -------------------------------------------------
import matplotlib.pyplot as plt

labels = ["Refresh now", "Investigate\n(rule silent)", "CTR gap\nonly", "Monitor"]
values = tier_counts.reindex(["1_refresh_now", "2_investigate_rule_silent", "3_ctr_gap_only", "4_monitor"]).to_numpy()
colors4 = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]

fig, ax = plt.subplots(figsize=(6.5, 4), facecolor="#fcfcfb")
ax.set_facecolor("#fcfcfb")
bars = ax.bar(labels, values, color=colors4, width=0.6)
for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width() / 2, val + 150, f"{val:,}\n({val / len(q):.0%})",
            ha="center", va="bottom", color="#0b0b0b", fontsize=9)
ax.set_ylabel("Pages", color="#0b0b0b")
ax.set_title("Action queue: how the 26,604 pages sort", color="#0b0b0b", fontsize=11)
ax.spines[["top", "right"]].set_visible(False)
ax.spines[["left", "bottom"]].set_color("#c3c2b7")
ax.tick_params(colors="#0b0b0b")
ax.set_ylim(0, max(values) * 1.25)
fig.text(0.5, -0.04, "42% of the queue is rule-silent rows the model still flags for investigation "
                       "-- the tier this paper's model adds on top of the hand rule.",
          ha="center", fontsize=9, color="#52514e", wrap=True)
fig.tight_layout()
fig.savefig("work/outputs/charts/action_tier_mix.svg", bbox_inches="tight", facecolor=fig.get_facecolor())
plt.close(fig)
print("wrote work/outputs/charts/action_tier_mix.svg")

--- action tier counts ---
action_tier
1_refresh_now                 5396
2_investigate_rule_silent    11276
3_ctr_gap_only                1730
4_monitor                     8202

tier 2 (rule silent, model flags) true-decline rate : 0.702


wrote work/outputs/capstone_action_queue.csv -- 26,604 rows ranked


wrote work/outputs/charts/action_tier_mix.svg


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Two charts (Sections 4 and 6) and two tables (the fold-level comparison in Section 4, the ranked
queue in Section 6) are what the deployed page needs. Collected and verified below rather than
regenerated -- everything already exists on disk from running this notebook top to bottom.

In [7]:
artifacts = [
    "work/outputs/charts/results_comparison.svg",
    "work/outputs/charts/action_tier_mix.svg",
    "work/outputs/w05_model_vs_baseline.csv",
    "work/outputs/capstone_action_queue.csv",
]
print("--- artifact manifest ---")
for a in artifacts:
    ok = os.path.exists(a)
    size = os.path.getsize(a) if ok else 0
    print(f"{'OK ' if ok else 'MISSING'} {a}  ({size:,} bytes)")

summary_table = summary.reset_index()
print("\n--- Results table (Section 4), for the paper ---")
print(summary_table.to_string(index=False))

--- artifact manifest ---
OK  work/outputs/charts/results_comparison.svg  (52,055 bytes)
OK  work/outputs/charts/action_tier_mix.svg  (49,925 bytes)
OK  work/outputs/w05_model_vs_baseline.csv  (2,056 bytes)
OK  work/outputs/capstone_action_queue.csv  (3,236,791 bytes)

--- Results table (Section 4), for the paper ---
                   arm   features  roc_auc  p@10  p@50  p@100  p@500
baseline (Week-4 rule)         --    0.532  0.76 0.808  0.782  0.746
          RandomForest permissive    0.645  0.92 0.844  0.842  0.787
          RandomForest     strict    0.611  0.84 0.868  0.812  0.771
                LogReg permissive    0.625  0.74 0.796  0.760  0.740
                LogReg     strict    0.586  0.72 0.724  0.706  0.704


## 8. Demo outline (5 minutes)

*One question, one method, one chart, one honest result, one recommendation.*

**0:00 - 0:45 | The question.** A content team has a few thousand published pages and cannot
review them all. Which ones should an editor open first? The two ways of being wrong do not cost
the same: a page flagged wrongly wastes an editor's hour, while a page missed keeps declining
until the client asks about it.

**0:45 - 1:45 | The method.** `26,604` pages from an anonymized FlyRank content extract, filtered
to those where decline is arithmetically possible. A RandomForest ranking model, validated on
five client-grouped folds so no client appears on both sides of a split, audited against five
leakage tests, and compared against a hand-written CTR-shortfall rule on the same folds.

**1:45 - 2:45 | The chart.** `work/outputs/charts/grouped_vs_random.svg`. The same model reads
`0.767` under a random split and `0.645` under a client-grouped one. The gap is not skill. It is
the model recognizing which client a row belongs to, which is worth nothing on a client it has
never seen. Running both is the only reason I know that.

**2:45 - 4:00 | The honest result.** The model reaches `0.645` mean ROC AUC against `0.532` for
the hand rule -- real separation from chance, and modest. Precision@50 is `0.844` on the same
folds. Two findings sit underneath it: zero-impression rows had to be removed before any of this
was measurable, because a page on zero cannot fall and `3,388` such rows were being scored as
confident non-declines; and the strict feature set scores lower on AUC while ranking slightly
better at the top of the queue, so both are reported rather than whichever flatters the story.

**4:00 - 5:00 | The recommendation.** Ship the ranked queue as decision-support for a human
editor, not as an automated pipeline. Work the top tier first, where the rule and the model
agree. The tier worth the most attention is the rule-silent one -- `42%` of the queue, rows the
hand rule never flags and the model still does, which is the part of the queue a rule alone
cannot produce. Do not read any of it as a causal claim that refreshing a flagged page fixes its
decline; no intervention was run here.


In [ ]:
# Chart for the demo and the social cut: what the split honesty cost.
# Uses auc_grouped / auc_random computed in Section 3 -- no numbers retyped.
import matplotlib.pyplot as plt

PAPER, INK, SIGNAL, CAUTION = "#FAF9F6", "#16181C", "#2B3D18", "#B4531A"
gap = auc_random - auc_grouped

fig, ax = plt.subplots(figsize=(7.4, 5.4), dpi=170)
fig.patch.set_facecolor(PAPER)
ax.set_facecolor(PAPER)

ax.bar([0, 1], [auc_grouped, auc_random], color=[SIGNAL, CAUTION], width=0.44, zorder=3)
for x, v in zip([0, 1], [auc_grouped, auc_random]):
    ax.text(x, v + 0.016, f"{v:.3f}", ha="center", fontsize=16, color=INK, zorder=4)

ax.plot([0.22, 0.78], [auc_grouped] * 2, ls=":", lw=1.1, color=INK, alpha=0.55, zorder=2)
ax.annotate("", xy=(0.5, auc_grouped), xytext=(0.5, auc_random),
            arrowprops=dict(arrowstyle="<->", color=CAUTION, lw=1.5), zorder=4)
ax.text(0.545, (auc_grouped + auc_random) / 2, f"{gap:.3f}",
        fontsize=13, color=CAUTION, va="center", zorder=4)

ax.axhline(0.5, ls="--", lw=1, color=INK, alpha=0.4, zorder=1)
ax.text(-0.44, 0.515, "chance (0.500)", fontsize=10.5, color=INK, alpha=0.6, zorder=4)

ax.set_xlim(-0.5, 1.5)
ax.set_ylim(0, 0.9)
ax.set_xticks([0, 1])
ax.set_xticklabels(["client-grouped split", "random split"], fontsize=12)
ax.set_ylabel("Mean ROC AUC", fontsize=13, color=INK)
ax.set_title(f"Same model, same {len(pop):,} pages, two validation designs",
             fontsize=14, color=SIGNAL, pad=16, loc="left")
ax.tick_params(labelsize=11, colors=INK, length=0)
for sp in ("top", "right", "left"):
    ax.spines[sp].set_visible(False)
ax.spines["bottom"].set_color(INK)
ax.spines["bottom"].set_alpha(0.3)

fig.text(0.115, 0.02,
         f"Holding out whole clients costs {gap:.3f} AUC. That gap is the model\n"
         "recognizing the client, not predicting decline.",
         fontsize=11, color=INK, alpha=0.72)
fig.subplots_adjust(top=0.87, bottom=0.21, left=0.115, right=0.97)

os.makedirs("work/outputs/charts", exist_ok=True)
fig.savefig("work/outputs/charts/grouped_vs_random.svg", facecolor=PAPER, bbox_inches="tight")
fig.savefig("work/outputs/charts/grouped_vs_random.png", facecolor=PAPER, bbox_inches="tight")
plt.close(fig)
print(f"wrote work/outputs/charts/grouped_vs_random.svg  (gap {gap:+.3f})")


## 9. Two shareable cuts

*The same work, retold for two audiences.*

### Social post (methodology)

> Most model posts report one number. Here are two for the same model.
>
> 0.645 ROC AUC when I hold out whole clients.
> 0.767 when I split rows at random.
>
> Same model. Same 26,604 pages. Same features.
>
> The 0.122 between them is not skill. It's the model recognizing which client a row belongs to
> -- worth nothing on a client it has never seen.
>
> I only know that because I ran both. If I'd run only the random split, I would have published
> 0.767 and believed it.
>
> The hand-written rule I had to beat scored 0.532.
>
> Built during the FlyRank AI internship on an anonymized content-performance extract -- 30,000
> pages across 32 client accounts. No client names, URLs, or search queries anywhere in it.
>
> Full method, five leakage tests, and the limitations -- link in the comments.

Chart: `work/outputs/charts/grouped_vs_random.png`.
Link: https://jericho-ram.github.io/FlyRank-Internship-ML/

### Employer summary (3 sentences)

> I built a ranked queue that scores which of a content team's published pages are declining in
> search, so an editor knows what to open first.
>
> It runs on an anonymized FlyRank content extract -- 30,000 rows across 32 client accounts,
> filtered to the 26,604 pages where decline is arithmetically possible.
>
> A random forest reached 0.645 ROC AUC across five client-grouped folds against 0.532 for the
> hand-written rule it had to beat -- a real but modest lift, and I report the grouped number
> rather than the 0.767 a random split would have given me.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.